# Static-Dynamic Decoupling Experiment

This notebook organizes the first implementation of the static/dynamic latent split for Kepler trajectory prediction.

The goal is to separate the latent representation into two parts:

- `z_static`: one trajectory-level latent vector, intended to remain constant throughout a trajectory.
- `z_dyn[t]`: one timestep-level latent vector, intended to capture time-varying state.

The implementation lives in:

- `model_static_dynamic.py`
- `kepler_static_dynamic.py`


## Architecture

The model predicts the next 2D position from a concatenation of static and dynamic latents:

```text
trajectory positions x[0:T]
        |
        +--> Static Encoder(first K points) --> z_static
        |
        +--> Causal Dynamic Transformer ------> z_dyn[t]

concat(z_static, z_dyn[t]) --> MLP Decoder --> x_hat[t+1]
```

First-pass configuration:

| Component | Value |
|---|---:|
| `static_window` | 20 |
| `static_dim` | 16 |
| `dynamic_dim` | 32 |
| static layers | 1 |
| dynamic layers | 2 |
| heads | 1 |
| parameters | 0.042242M |

The static vector is constant by construction: it is inferred once from the initial window and broadcast across timesteps.

## Probe Design

The key diagnostic is a leakage matrix.

| Source latent | Static orbital targets | Dynamic local targets |
|---|---|---|
| `z_static` | should be high | should be low |
| `z_dyn[t]` | should be low | should be high |

Static targets include orbital parameters such as `a`, `b`, `e`, `LRL_x`, `LRL_y`, and orientation. Dynamic targets include `x`, `y`, `r`, `Fx`, `Fy`, and force direction.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

RESULT_PATH = "results/kepler_static_dynamic/static_dynamic_num_trajectories_100_steps_101_seed_1.npz"
RESULT_PATH

'results/kepler_static_dynamic/static_dynamic_num_trajectories_100_steps_101_seed_1.npz'

## Reproduce the First Run

The initial CPU-scale experiment was run with:

```bash
python kepler_static_dynamic.py \
  --num_trajectories 100 \
  --n_steps 101 \
  --batch_size 64 \
  --prob_freq 50 \
  --log_freq 25 \
  --probe_sample_size 100 \
  --rollout_sample_size 100 \
  --static_window 20 \
  --static_dim 16 \
  --dynamic_dim 32 \
  --n_layer_static 1 \
  --n_layer_dynamic 2 \
  --seed 1
```

In [ ]:
# Uncomment to rerun the first experiment from this notebook.
# This uses the existing data_cv folder and writes to results/kepler_static_dynamic/.

# !python kepler_static_dynamic.py \
#   --num_trajectories 100 \
#   --n_steps 101 \
#   --batch_size 64 \
#   --prob_freq 50 \
#   --log_freq 25 \
#   --probe_sample_size 100 \
#   --rollout_sample_size 100 \
#   --static_window 20 \
#   --static_dim 16 \
#   --dynamic_dim 32 \
#   --n_layer_static 1 \
#   --n_layer_dynamic 2 \
#   --seed 1

In [ ]:
loaded = np.load(RESULT_PATH, allow_pickle=True)

final_train_loss = float(loaded["final_train_loss"])
final_test_loss = float(loaded["final_test_loss"])
eval_steps = list(loaded["eval_steps"])
final_probe_summary = loaded["final_probe_summary"].item()
final_error_stats_train = loaded["final_error_stats_train"].item()
final_error_stats_test = loaded["final_error_stats_test"].item()
eval_results = list(loaded["eval_results"])

print(f"final_train_loss: {final_train_loss:.6f}")
print(f"final_test_loss:  {final_test_loss:.6f}")
print(f"eval_steps:       {eval_steps}")
print()
print(f"train rollout mean error: {final_error_stats_train['mean_error']:.6f}")
print(f"test rollout mean error:  {final_error_stats_test['mean_error']:.6f}")
print(f"test rollout R2 x:        {final_error_stats_test['mean_r2_x']:.6f}")
print(f"test rollout R2 y:        {final_error_stats_test['mean_r2_y']:.6f}")
print()
final_probe_summary

## First Result Summary

| Metric | Value |
|---|---:|
| final train loss | 0.037792 |
| final test loss | 0.032935 |
| test rollout mean error | 1.471020 |
| test rollout R² x | -2.182489 |
| test rollout R² y | -1.830513 |

The one-step prediction loss improved quickly, but rollout quality is still poor at this small scale. This is not surprising for a 101-step CPU smoke run; the more important early result is the probe/leakage behavior.

In [ ]:
leakage_matrix = np.array([
    [final_probe_summary["static_from_z_static_mean_r2"], final_probe_summary["dynamic_from_z_static_repeated_mean_r2"]],
    [final_probe_summary["static_from_z_dyn_last_mean_r2"], final_probe_summary["dynamic_from_z_dyn_mean_r2"]],
])

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(leakage_matrix, vmin=0, vmax=1, cmap="viridis")

ax.set_xticks([0, 1], labels=["static targets", "dynamic targets"])
ax.set_yticks([0, 1], labels=["z_static", "z_dyn"])
ax.set_title("Mean linear-probe R² leakage matrix")

for i in range(leakage_matrix.shape[0]):
    for j in range(leakage_matrix.shape[1]):
        ax.text(j, i, f"{leakage_matrix[i, j]:.3f}", ha="center", va="center", color="white")

fig.colorbar(im, ax=ax, label="mean R²")
plt.tight_layout()
plt.show()

## Leakage Interpretation

Final probe summary:

| Probe | Mean R² |
|---|---:|
| static targets from `z_static` | 0.6722 |
| static targets from `z_dyn_last` | 0.8832 |
| dynamic targets from `z_dyn[t]` | 0.5800 |
| dynamic targets from repeated `z_static` | 0.2104 |

Interpretation:

- `z_static` does encode useful orbital information.
- `z_dyn[t]` also encodes dynamic state, as desired.
- But `z_dyn_last` leaks even more static orbital information than `z_static` in this first run.
- So the model is mechanically functional, but the decoupling is not yet clean.

In [ ]:
final_probe_results = eval_results[-1]["probe_results"]

def top_items(section, n=8):
    return sorted(final_probe_results[section].items(), key=lambda kv: kv[1], reverse=True)[:n]

print("Top static targets from z_static")
for name, score in top_items("static_from_z_static"):
    print(f"  {name:18s} {score:.4f}")

print("\nTop static targets leaked through z_dyn_last")
for name, score in top_items("static_from_z_dyn_last"):
    print(f"  {name:18s} {score:.4f}")

print("\nTop dynamic targets from z_dyn")
for name, score in top_items("dynamic_from_z_dyn"):
    print(f"  {name:18s} {score:.4f}")

print("\nTop dynamic leakage through repeated z_static")
for name, score in top_items("dynamic_from_z_static_repeated"):
    print(f"  {name:18s} {score:.4f}")

## Next Experiment Ideas

The first useful failure mode is static leakage through `z_dyn`. Natural next steps:

1. Add a stronger static bottleneck or increase pressure to route orbit-level information through `z_static`.
2. Add an adversarial/static-leakage penalty that discourages static orbital parameters from being linearly decodable from `z_dyn[t]`.
3. Add static-window consistency by encoding multiple windows from the same trajectory and penalizing disagreement.
4. Compare against the baseline `GPTCV` under the same training budget and probe protocol.

The static-window sliding issue can be handled later by sampling multiple windows per trajectory and treating their static encodings as views of the same orbit.